In [ ]:
# HYPSTER_HOST_CELL:create-auto
from pathlib import Path
from uuid import uuid4

import hypster
from hypster import HP

hypster_path = Path(hypster.__file__).resolve()
assert "site-packages" in hypster_path.parts, f"expected installed wheel import, got {hypster_path}"
print(f"HYPSTER_IMPORT_PATH={hypster_path}; EXECUTION_ID={uuid4().hex}")


def remote(hp: HP):
    return {"temperature": hp.float(0.25, name="temperature", min=0.0, max=2.0)}


def config(hp: HP):
    mode = hp.select(["local", "remote"], name="mode", default="local")
    if mode == "remote":
        return {"mode": mode, "remote": hp.nest(remote, name="remote")}
    return {"mode": mode}


result = hypster.interact(config)
result

In [ ]:
# HYPSTER_HOST_CELL:verify-auto
expected = {"mode": "remote", "remote.temperature": 1.25}
assert result.params == expected, f"{result.params!r} != {expected!r}"
print(f"HYPSTER_PARAMS_VERIFIED={result.params!r}; EXECUTION_ID={uuid4().hex}")

In [ ]:
# HYPSTER_HOST_CELL:create-manual
manual_result = hypster.interact(config, auto_apply=False)
print(f"HYPSTER_MANUAL_READY; EXECUTION_ID={uuid4().hex}")
manual_result

In [ ]:
# HYPSTER_HOST_CELL:verify-manual-staged
expected = {"mode": "local"}
assert manual_result.params == expected, f"{manual_result.params!r} != {expected!r}"
print(f"HYPSTER_MANUAL_STAGED_VERIFIED={manual_result.params!r}; EXECUTION_ID={uuid4().hex}")

In [ ]:
# HYPSTER_HOST_CELL:verify-manual-applied
expected = {"mode": "remote", "remote.temperature": 1.25}
assert manual_result.params == expected, f"{manual_result.params!r} != {expected!r}"
print(f"HYPSTER_MANUAL_APPLIED_VERIFIED={manual_result.params!r}; EXECUTION_ID={uuid4().hex}")

In [ ]:
# HYPSTER_HOST_CELL:verify-manual-reset
expected = {"mode": "local"}
assert manual_result.params == expected, f"{manual_result.params!r} != {expected!r}"
print(f"HYPSTER_MANUAL_RESET_VERIFIED={manual_result.params!r}; EXECUTION_ID={uuid4().hex}")

In [ ]:
# HYPSTER_HOST_CELL:create-protocol
from IPython.display import display

protocol_result = hypster.interact(config)
protocol_widget = protocol_result.interact()
protocol_params_before = protocol_result.params
protocol_action_before = dict(protocol_widget.action)
display(protocol_widget)
print(f"HYPSTER_PROTOCOL_READY; EXECUTION_ID={uuid4().hex}")

In [ ]:
# HYPSTER_HOST_CELL:inject-protocol-mismatch
mismatched_snapshot = dict(protocol_result.snapshot)
mismatched_snapshot["protocol_version"] = 2
protocol_widget.snapshot = mismatched_snapshot
print(f"HYPSTER_PROTOCOL_MISMATCH_SENT=2; EXECUTION_ID={uuid4().hex}")

In [ ]:
# HYPSTER_HOST_CELL:verify-protocol-unchanged
assert protocol_widget.action == protocol_action_before == {}, protocol_widget.action
assert protocol_result.params == protocol_params_before == {"mode": "local"}
print(
    f"HYPSTER_PROTOCOL_UNCHANGED={protocol_result.params!r}; "
    f"ACTION={protocol_widget.action!r}; EXECUTION_ID={uuid4().hex}"
)

In [ ]:
# HYPSTER_HOST_CELL:recover-protocol
protocol_widget.snapshot = protocol_result.snapshot
print(f"HYPSTER_PROTOCOL_RECOVERED=1; EXECUTION_ID={uuid4().hex}")